# Titanic Survival Analysis — Full Portfolio Notebook
### End-to-End EDA, Multivariate Analysis & Feature Engineering
**Dataset:** Kaggle Titanic Training Set (train.csv) — 891 passengers, real historical survival labels  
**Central question:** What combination of passenger characteristics best predicts survival?  
**Last updated:** Session 2 — Multivariate analysis complete

---
### Notebook structure
1. Setup
2. Load data
3. Raw data overview & missing values
4. Missing data imputation
5. Feature engineering
6. EDA — survival rates by factor
7. Sex × Class heatmap
8. Scatter plots — each variable vs survival
9. KDE — Age and Fare distributions
10. Correlation matrix — multicollinearity detection
11. Covariance matrix
12. Joint distribution — Age × log(Fare)
13. Clean feature set for modeling
14. What's next


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('Set2')

print("✅ Libraries loaded")

## 2. Load data
Upload `train.csv` from Kaggle → https://www.kaggle.com/c/titanic/data

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
df_raw = pd.read_csv(io.BytesIO(uploaded['train.csv']))
print(f"Shape: {df_raw.shape}")
df_raw.head(10)

**Alternative: mount Google Drive (recommended — persists across sessions)**

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/titanic/train.csv')
# print(f"Shape: {df_raw.shape}")

## 3. Raw data overview

In [ ]:
print("=== DATA TYPES ===")
print(df_raw.dtypes)
print("\n=== MISSING VALUES ===")
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(1)
print(pd.DataFrame({'missing': missing, 'pct_%': pct})[missing > 0])
print("\n=== DESCRIPTIVE STATS ===")
df_raw.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
missing_cols = {'Age': 19.9, 'Cabin': 77.1, 'Embarked': 0.2}
colors = ['#D85A30','#D85A30','#1D9E75']
axes[0].barh(list(missing_cols.keys()), list(missing_cols.values()), color=colors, alpha=0.85)
axes[0].set_xlabel('% missing')
axes[0].set_title('Missing values by column', fontweight='bold')
axes[0].axvline(20, color='gray', linestyle='--', linewidth=0.8, label='20% threshold')
axes[0].legend(fontsize=9)

df_raw['Survived'].value_counts().plot(kind='bar', ax=axes[1], color=['#D85A30','#1D9E75'], alpha=0.85)
axes[1].set_title('Survival distribution (0=died, 1=survived)', fontweight='bold')
axes[1].set_xticklabels(['Died','Survived'], rotation=0)
axes[1].set_ylabel('Count')

df_raw['Pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color=['#378ADD','#1D9E75','#D85A30'], alpha=0.85)
axes[2].set_title('Passenger class distribution', fontweight='bold')
axes[2].set_xticklabels(['1st','2nd','3rd'], rotation=0)

plt.suptitle('Dataset overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"Overall survival rate: {df_raw['Survived'].mean()*100:.1f}%")

## 4. Missing data imputation
**Strategy:**
- `Age` (20% missing) → median by title group — preserves demographic signal
- `Cabin` (77% missing) → binary flag `Has_cabin` — missingness is the signal
- `Embarked` (2 missing) → fill with mode (Southampton)

**Why title-group imputation for Age?** A "Master" is almost always a boy under 13. A "Mrs" is typically 30s–40s. Using a global median would assign age 28 to everyone — systematically wrong for these groups.

In [ ]:
df = df_raw.copy()

# Extract title
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.', expand=False)
rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
df['Title'] = df['Title'].replace(rare, 'Other')
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

print("=== TITLE DISTRIBUTION ===")
print(df['Title'].value_counts())

print("\n=== AGE MEDIANS BY TITLE ===")
title_medians = df.groupby('Title')['Age'].median()
print(title_medians.round(1))

In [ ]:
# Impute Age
df['Age'] = df.apply(
    lambda r: title_medians[r['Title']] if pd.isna(r['Age']) else r['Age'], axis=1)

# Cabin flag
df['Has_cabin'] = df['Cabin'].notna().astype(int)

# Embarked
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Verify
print("Missing after imputation:")
print(df[['Age','Embarked','Has_cabin']].isnull().sum())

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df_raw['Age'].dropna().plot.kde(ax=axes[0], label='Before (n=714)', color='#D85A30', linewidth=2)
df['Age'].plot.kde(ax=axes[0], label='After (n=891)', color='#1D9E75', linewidth=2, linestyle='--')
axes[0].set_title('Age: before vs after imputation', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].legend()

surv_cabin = df.groupby('Has_cabin')['Survived'].mean() * 100
axes[1].bar(['No cabin record','Has cabin record'], surv_cabin.values, 
            color=['#D85A30','#1D9E75'], alpha=0.85)
axes[1].axhline(df['Survived'].mean()*100, color='gray', linestyle='--', label=f"Overall {df['Survived'].mean()*100:.1f}%")
axes[1].set_ylabel('Survival rate (%)'); axes[1].set_title('Has cabin → survival rate', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Feature engineering
Creating features that expose hidden signal the raw columns don't capture directly.

In [ ]:
# Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['FamilyGroup'] = pd.cut(df['FamilySize'],
    bins=[0,1,3,5,20], labels=['Alone','Small','Medium','Large'])

# Fare transformation
df['LogFare'] = np.log1p(df['Fare'])

# Age features
df['IsChild']  = (df['Age'] < 10).astype(int)
df['AgeGroup'] = pd.cut(df['Age'],
    bins=[0,12,18,35,60,100], labels=['Child','Teen','YoungAdult','Adult','Senior'])

# Interaction
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
df['Sex_num']    = (df['Sex'] == 'male').astype(int)

print("New features added:")
new_feats = ['Title','FamilySize','IsAlone','FamilyGroup','LogFare','IsChild','AgeGroup','Sex_Pclass','Sex_num','Has_cabin']
for f in new_feats:
    print(f"  {f}: {df[f].dtype} | unique={df[f].nunique()}")

## 6. EDA — survival rates by each factor

In [ ]:
overall = df['Survived'].mean()
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

factors = [('Sex','Sex'),('Pclass','Passenger class'),('IsChild','Is Child (<10)'),
           ('IsAlone','Traveling alone'),('FamilyGroup','Family group'),('AgeGroup','Age group')]

for ax, (col, title) in zip(axes, factors):
    rates = df.groupby(col, observed=True)['Survived'].mean().sort_values(ascending=False)
    colors = ['#1D9E75' if r >= overall else '#D85A30' for r in rates]
    bars = ax.bar(rates.index.astype(str), rates.values * 100, color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(overall*100, color='#888', linestyle='--', linewidth=1.2, label=f'Overall {overall:.1%}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Survival rate (%)')
    ax.set_ylim(0, 110)
    for bar, r in zip(bars, rates.values):
        ax.text(bar.get_x()+bar.get_width()/2, r*100+2, f'{r:.1%}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Survival rates by factor\n(teal ≥ average, red < average)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 7. Sex × Class interaction — real historical rates

In [ ]:
pivot  = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='mean')
counts = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='count')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
            linewidths=0.5, annot=False, square=True)

for i, sex in enumerate(pivot.index):
    for j, cls in enumerate(pivot.columns):
        rate = pivot.loc[sex, cls]
        n    = int(counts.loc[sex, cls])
        color = 'white' if rate > 0.6 or rate < 0.25 else 'black'
        ax.text(j+0.5, i+0.38, f'{rate:.1%}', ha='center', va='center',
                fontsize=14, fontweight='bold', color=color)
        ax.text(j+0.5, i+0.65, f'n={n}', ha='center', va='center',
                fontsize=10, color=color)

ax.set_title('Survival rate by Sex × Class\n(real historical rates — not the clean 100%/0% of test set)',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Passenger class'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

print("Key insight: 3rd class women → 50% survival. 1st class men → 37%.")
print("Sex dominates, but class still matters significantly for women.")

## 8. Scatter plots — each variable vs survival

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
surv  = df[df['Survived']==1]
nsurv = df[df['Survived']==0]
np.random.seed(42)

plots = [('Age','Age','r=−0.065'),('LogFare','log(Fare+1)','r=+0.330'),
         ('FamilySize','Family size','r=+0.017'),('Pclass','Passenger class','r=−0.338'),
         ('Sex_num','Sex (1=male)','r=−0.543'),('IsChild','Is child','r=+0.106')]

for ax, (col, xlabel, r_str) in zip(axes, plots):
    jit = lambda n, amt: np.random.uniform(-amt, amt, n)
    amt = max(df[col].std() * 0.08, 0.1)
    ax.scatter(surv[col]  + jit(len(surv), amt),  np.ones(len(surv))*0.72  + jit(len(surv), 0.15),
               alpha=0.35, color='#1D9E75', s=20, label='Survived')
    ax.scatter(nsurv[col] + jit(len(nsurv), amt), np.ones(len(nsurv))*0.28 + jit(len(nsurv), 0.15),
               alpha=0.25, color='#D85A30', s=16, label='Not survived')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_yticks([0.28, 0.72])
    ax.set_yticklabels(['Not survived','Survived'], fontsize=9)
    ax.set_title(f'{xlabel}\n{r_str}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 1)

plt.suptitle('Jittered scatter — each variable vs survival', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 9. Kernel Density Estimation (KDE)
KDE places a small bell curve on each data point and sums them → smooth continuous density estimate.  
**Reading these:** where the teal (survived) and red (not survived) curves separate = predictive signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
surv  = df[df['Survived']==1]
nsurv = df[df['Survived']==0]
age_range  = np.linspace(0, 80, 200)
fare_range = np.linspace(0, 7, 200)

for grp, color, label in [(surv,'#1D9E75','Survived'),(nsurv,'#D85A30','Not survived')]:
    kde_a = gaussian_kde(grp['Age'].dropna(), bw_method='scott')
    kde_f = gaussian_kde(grp['LogFare'].dropna(), bw_method='scott')
    axes[0].plot(age_range,  kde_a(age_range),  color=color, label=label, linewidth=2.5)
    axes[0].fill_between(age_range, kde_a(age_range), alpha=0.12, color=color)
    axes[1].plot(fare_range, kde_f(fare_range), color=color, label=label, linewidth=2.5)
    axes[1].fill_between(fare_range, kde_f(fare_range), alpha=0.12, color=color)

axes[0].axvline(10, color='gray', linestyle=':', linewidth=1.5, label='IsChild threshold (age 10)')
axes[0].set_title('Age KDE by survival', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Density'); axes[0].legend()
axes[0].text(3, axes[0].get_ylim()[1]*0.8, 'Child\nsurvival\nbump', fontsize=9, color='gray', ha='center')

axes[1].set_title('log(Fare+1) KDE by survival', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(Fare+1)'); axes[1].set_ylabel('Density'); axes[1].legend()

plt.suptitle('KDE: shape of distributions by survival outcome', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print("Therefore:")
print("  Age   → weak overall signal; child (<10) bump is real → engineer IsChild")
print("  Fare  → strong separation; log-transform justified → confirmed LogFare")

## 10. Correlation matrix — multicollinearity detection

In [ ]:
model_cols   = ['Survived','Sex_num','Pclass','LogFare','Age','FamilySize','IsAlone','IsChild','SibSp','Parch']
col_labels   = ['Survived','Sex','Pclass','LogFare','Age','FamilySize','IsAlone','IsChild','SibSp','Parch']

corr = df[model_cols].corr()
corr.index = corr.columns = col_labels

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, annot_kws={'size':9}, square=True)
axes[0].set_title('Pearson Correlation Matrix', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

surv_corr = corr['Survived'].drop('Survived').sort_values(key=abs, ascending=True)
colors = ['#1D9E75' if v > 0 else '#D85A30' for v in surv_corr]
bars = axes[1].barh(surv_corr.index, surv_corr.values, color=colors, alpha=0.85)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(0.3,  color='#1D9E75', linestyle='--', linewidth=1, alpha=0.5)
axes[1].axvline(-0.3, color='#1D9E75', linestyle='--', linewidth=1, alpha=0.5, label='|r|=0.3')
for bar, v in zip(bars, surv_corr.values):
    axes[1].text(v + (0.01 if v>=0 else -0.01), bar.get_y()+bar.get_height()/2,
                 f'{v:+.3f}', va='center', ha='left' if v>=0 else 'right', fontsize=10)
axes[1].set_title('Correlation with Survival (ranked)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pearson r'); axes[1].legend()

plt.tight_layout(); plt.show()

print("\n=== MULTICOLLINEARITY FLAGS (|r| > 0.5) ===")
for i in range(len(col_labels)):
    for j in range(i+1, len(col_labels)):
        r = corr.iloc[i,j]
        if abs(r) > 0.5:
            action = 'DROP one' if abs(r) > 0.7 else 'WATCH'
            print(f"  {col_labels[i]:12s} × {col_labels[j]:12s}: r={r:+.3f}  → {action}")

## 11. Covariance matrix
Covariance = correlation in original units. Age (σ≈13) inflates all its covariances.  
**Rule:** correlation for interpretation; covariance for PCA and distance algorithms.

In [ ]:
clean_cols   = ['Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
clean_labels = ['Sex','Pclass','LogFare','Age','IsAlone','IsChild']

cov = df[clean_cols].cov()
cov.index = cov.columns = clean_labels

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(cov, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[0], linewidths=0.5, annot_kws={'size':9}, square=True)
axes[0].set_title('Covariance Matrix (clean feature set)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

pairs = ['Age×LogFare','Age×IsChild','IsAlone×LogFare','Pclass×LogFare']
cov_vals  = [1.40, -1.761, -0.227, -0.536]
corr_vals = [0.111, -0.531, -0.478, -0.661]
x = np.arange(len(pairs)); w = 0.35

axes[1].bar(x-w/2, [abs(v) for v in cov_vals],  w, label='|Covariance|',  color='#378ADD', alpha=0.8)
axes[1].bar(x+w/2, [abs(v) for v in corr_vals], w, label='|Correlation|', color='#1D9E75', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(pairs, rotation=15, ha='right', fontsize=9)
axes[1].set_title('Covariance vs Correlation magnitude\n(same pairs — scale difference is the point)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Absolute value'); axes[1].legend()
axes[1].text(0, 1.5, 'Age dominates covariance\n(σ_Age ≈ 13)', fontsize=9, color='#D85A30',
             bbox=dict(boxstyle='round', facecolor='#FAECE7', alpha=0.8))

plt.tight_layout(); plt.show()

## 12. Joint distribution — Age × log(Fare)
The joint distribution shows how two variables behave *simultaneously*.  
Marginals alone can miss structure that only appears in 2D.

In [ ]:
from scipy.stats import gaussian_kde as gkde

surv  = df[df['Survived']==1][['Age','LogFare']].dropna()
nsurv = df[df['Survived']==0][['Age','LogFare']].dropna()

age_grid  = np.linspace(0, 80, 60)
fare_grid = np.linspace(0, 7, 60)
A, F = np.meshgrid(age_grid, fare_grid)
pos  = np.vstack([A.ravel(), F.ravel()])

kde_s  = gkde(surv.values.T,  bw_method='scott')
kde_ns = gkde(nsurv.values.T, bw_method='scott')
Z_s    = kde_s(pos).reshape(60,60)
Z_ns   = kde_ns(pos).reshape(60,60)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, Z, pts, color, title in [
    (axes[0], Z_ns, nsurv, '#D85A30', 'Not survived'),
    (axes[1], Z_s,  surv,  '#1D9E75', 'Survived'),
]:
    contour = ax.contourf(A, F, Z, levels=8, cmap='Oranges' if color=='#D85A30' else 'Greens', alpha=0.7)
    ax.contour(A, F, Z, levels=5, colors=color, linewidths=0.8, alpha=0.6)
    ax.scatter(pts['Age'], pts['LogFare'], alpha=0.25, color=color, s=10)
    ax.set_xlabel('Age'); ax.set_ylabel('log(Fare+1)')
    ax.set_title(f'Joint density — {title}', fontweight='bold')
    plt.colorbar(contour, ax=ax, shrink=0.8)

axes[2].scatter(nsurv['Age'], nsurv['LogFare'], alpha=0.2, color='#D85A30', s=12, label='Not survived')
axes[2].scatter(surv['Age'],  surv['LogFare'],  alpha=0.3, color='#1D9E75', s=12, label='Survived')
axes[2].contour(A, F, Z_ns, levels=4, colors='#D85A30', linewidths=1.2, alpha=0.7)
axes[2].contour(A, F, Z_s,  levels=4, colors='#1D9E75', linewidths=1.2, alpha=0.7)
axes[2].set_xlabel('Age'); axes[2].set_ylabel('log(Fare+1)')
axes[2].set_title('Overlay — both groups', fontweight='bold')
axes[2].legend(fontsize=9)

r_s,  _ = stats.pearsonr(surv['Age'],  surv['LogFare'])
r_ns, _ = stats.pearsonr(nsurv['Age'], nsurv['LogFare'])
r_all,_ = stats.pearsonr(df['Age'].dropna(), df.loc[df['Age'].notna(),'LogFare'])
print(f"Correlations — Overall: r={r_all:.3f}  |  Survived: r={r_s:.3f}  |  Not survived: r={r_ns:.3f}")
print("\nKey insight: survivor r is DOUBLE non-survivor r.")
print("Dual-peak structure: low-fare women/children + high-fare 1st class — two distinct survivor groups.")

plt.suptitle('Joint distribution: Age × log(Fare)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 13. Clean feature set — modeling decisions

Based on correlation analysis, KDE, and joint distribution findings:

In [ ]:
print("╔══════════════════════════════════════════════════════════════════╗")
print("║                FEATURE SELECTION DECISIONS                       ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║ KEEP                                                             ║")
keep = {
    'Sex_num':  'r=−0.543  strongest single predictor',
    'Pclass':   'r=−0.338  class signal; independent from fare',
    'LogFare':  'r=+0.330  wealth signal; log-transform justified by KDE',
    'IsAlone':  'r=−0.203  solo travelers survived worst',
    'IsChild':  'r=+0.106  captures the age<10 KDE bump explicitly',
    'Age':      'r=−0.065  weak linear but non-linear; keep as secondary',
}
for f, reason in keep.items():
    print(f"║  ✅ {f:12s} {reason}")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║ DROP — multicollinearity                                         ║")
drop = {
    'FamilySize': 'r=+0.891 with SibSp; IsAlone captures it better',
    'SibSp':      'r=+0.891 with FamilySize; redundant',
    'Parch':      'r=+0.783 with FamilySize; redundant',
}
for f, reason in drop.items():
    print(f"║  ❌ {f:12s} {reason}")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║ CONSIDER — interaction term                                      ║")
print("║  🔄 Age×LogFare  r_survived=+0.221 vs r_not=+0.078 → dual peak  ║")
print("╚══════════════════════════════════════════════════════════════════╝")

model_features = ['Survived','Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
df_model = df[model_features].copy()
df_model.columns = ['Survived','Sex','Pclass','LogFare','Age','IsAlone','IsChild']
print(f"\nModel-ready dataset: {df_model.shape}")
print(df_model.describe().round(3))

## 14. What's next

### Completed ✅
1. EDA — survival rates across all factors  
2. Missing data — imputation by title group (Age), binary flag (Cabin)  
3. Feature engineering — Title, IsChild, IsAlone, LogFare, FamilySize, Sex_Pclass  
4. Distribution theory — KDE, normality tests (KS, Shapiro-Wilk), transforms (Box-Cox, Yeo-Johnson)  
5. Scatter plots — visual separation by variable  
6. Correlation matrix — multicollinearity detection  
7. Covariance matrix — scale effects, when to use each  
8. Joint distribution — Age × LogFare; dual-peak survivor structure  

### In progress / next ⬜
- Copula analysis — dependency structure beyond linear correlation  
- PCA — dimensionality reduction, variance explained  
- Model building — Logistic Regression → Random Forest → XGBoost  
- Model evaluation — confusion matrix, AUC-ROC, feature importance  
- Portfolio artifacts — case study PDF, GitHub README  

### Director-level reminder
The clean feature set is: **Sex, Pclass, LogFare, Age, IsAlone, IsChild**  
Before modeling, verify these decisions hold in cross-validation.  
The Sex × Pclass interaction is the single most powerful combined signal — consider it as an explicit feature.  
The Age × LogFare interaction (r_surv=+0.221 vs r_ns=+0.078) is the key joint distribution finding — worth including.


## 15. Copula analysis — dependency structure beyond linear correlation
A copula separates a joint distribution into:
1. **Marginal distributions** (each variable's own shape)
2. **Dependency structure** (how they co-move, independent of their shapes)

This lets us compare dependency structures across groups (survivors vs not) without being confused by different marginal distributions.

In [ ]:
from scipy.stats import rankdata, kendalltau, spearmanr, norm as scipy_norm

def to_uniform(x):
    """Rank transform to uniform [0,1] — strips marginal distributions"""
    return rankdata(x) / (len(x) + 1)

age_vals  = df_model['Age'].values
fare_vals = df_model['LogFare'].values

# Step 1: uniform marginals
u_age  = to_uniform(age_vals)
u_fare = to_uniform(fare_vals)

# Step 2: Gaussian copula — transform to normal quantiles, measure correlation
z_age  = scipy_norm.ppf(u_age)
z_fare = scipy_norm.ppf(u_fare)
gauss_rho = np.corrcoef(z_age, z_fare)[0,1]

# Dependency measures
tau, p_tau = kendalltau(age_vals, fare_vals)
spear_rho, _ = spearmanr(age_vals, fare_vals)
theta_gumbel  = 1 / (1 - tau)   # Gumbel copula parameter
theta_clayton = 2 * tau / (1 - tau)  # Clayton copula parameter

print(f"=== COPULA PARAMETERS (Age × LogFare) ===")
print(f"Pearson r:         {np.corrcoef(age_vals, fare_vals)[0,1]:.4f}  (linear correlation)")
print(f"Spearman rho:      {spear_rho:.4f}  (rank correlation)")
print(f"Kendall tau:       {tau:.4f}  (concordance-discordance)  p={p_tau:.4f}")
print(f"Gaussian copula rho: {gauss_rho:.4f}  (dependency in normal space)")
print(f"Gumbel theta:      {theta_gumbel:.4f}  (>1 = positive upper tail dep.)")
print(f"Clayton theta:     {theta_clayton:.4f}  (>0 = positive lower tail dep.)")
print(f"\nGumbel theta=1.095 → near-independence (theta=1 is complete independence)")

In [ ]:
# Bivariate copula: compare dependency structure by survival group
print("=== BIVARIATE COPULA — SURVIVED vs NOT SURVIVED ===")
for name, mask in [('Survived', df_model['Survived']==1), ('Not survived', df_model['Survived']==0)]:
    grp = df_model[mask]
    a, f = grp['Age'].values, grp['LogFare'].values
    u1, u2 = to_uniform(a), to_uniform(f)
    z1, z2 = scipy_norm.ppf(u1), scipy_norm.ppf(u2)
    rho  = np.corrcoef(z1, z2)[0,1]
    tau_g, _ = kendalltau(a, f)
    print(f"  {name:15s}: Gaussian rho={rho:.4f}, Kendall tau={tau_g:.4f}, n={len(grp)}")

print("\nKey insight: dependency is 5x stronger among survivors.")
print("Age and wealth were tightly coupled for survivors — older = wealthier.")
print("Non-survivors were uniformly poor regardless of age — near-zero coupling.")

# Plot: uniform marginals scatter colored by survival
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# All passengers
axes[0].scatter(u_age[df_model['Survived']==0], u_fare[df_model['Survived']==0],
                alpha=0.3, color='#D85A30', s=12, label='Not survived')
axes[0].scatter(u_age[df_model['Survived']==1], u_fare[df_model['Survived']==1],
                alpha=0.4, color='#1D9E75', s=14, label='Survived')
axes[0].set_xlabel('U(Age)'); axes[0].set_ylabel('U(LogFare)')
axes[0].set_title('Copula space — uniform marginals\n(dependency structure only)', fontweight='bold')
axes[0].legend(fontsize=9)

# Bivariate by group
for i, (name, mask, color) in enumerate([
    ('Survived',     df_model['Survived']==1, '#1D9E75'),
    ('Not survived', df_model['Survived']==0, '#D85A30')
]):
    grp = df_model[mask]
    u1 = to_uniform(grp['Age'].values)
    u2 = to_uniform(grp['LogFare'].values)
    axes[i+1].scatter(u1, u2, alpha=0.4, color=color, s=14)
    rho = np.corrcoef(scipy_norm.ppf(u1), scipy_norm.ppf(u2))[0,1]
    axes[i+1].set_xlabel('U(Age)'); axes[i+1].set_ylabel('U(LogFare)')
    axes[i+1].set_title(f'{name}\nGaussian ρ={rho:.3f}', fontweight='bold')
    axes[i+1].set_facecolor('#f8f8f8' if name=='Not survived' else '#f0faf5')

plt.suptitle('Copula analysis: dependency structure by survival group',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 16. PCA — principal component analysis
PCA finds the directions of maximum variance in the feature space.  
Each principal component is a linear combination of features, ordered by variance explained.  
**Use cases:** dimensionality reduction, visualization, multicollinearity resolution, feature importance insight.

In [ ]:
from numpy.linalg import eig

features = ['Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
X = df_model[features].values
survived_pca = df_model['Survived'].values

# Standardize (essential for PCA — equalizes scales)
Xm = (X - X.mean(axis=0)) / X.std(axis=0)

# Eigendecomposition of covariance matrix
cov_mat = np.cov(Xm.T)
eigenvalues, eigenvectors = eig(cov_mat)
order = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[order].real
eigenvectors = eigenvectors[:, order].real

explained_var = eigenvalues / eigenvalues.sum()
cumulative    = np.cumsum(explained_var)

print("=== EIGENVALUES & VARIANCE EXPLAINED ===")
print(f"{'PC':>4} {'Eigenvalue':>12} {'Explained':>12} {'Cumulative':>12} {'Keep?':>8}")
print("-" * 55)
for i, (ev, exp, cum) in enumerate(zip(eigenvalues, explained_var, cumulative)):
    keep = "✅ Yes" if ev > 1.0 else "—"
    print(f"PC{i+1:>2} {ev:>12.4f} {exp*100:>11.1f}% {cum*100:>11.1f}% {keep:>8}")

print("\nKaiser rule: keep components with eigenvalue > 1 → PC1 and PC2")
print(f"2 PCs explain {cumulative[1]*100:.1f}% of variance")
print(f"3 PCs explain {cumulative[2]*100:.1f}% of variance")

In [ ]:
# Loadings table
print("=== LOADINGS (how each feature contributes to each PC) ===")
print(f"{'Feature':>12} {'PC1':>8} {'PC2':>8} {'PC3':>8} {'PC4':>8}")
print("-" * 48)
for j, f in enumerate(features):
    vals = [f"{eigenvectors[j,i]:+.3f}" for i in range(4)]
    print(f"{f:>12} {'  '.join(vals)}")

print("\nInterpretation:")
print("  PC1: high LogFare (−), high Pclass (+), IsAlone (+)  → SOCIOECONOMIC axis")
print("  PC2: high Age (+), IsChild (−)                       → AGE axis")
print("  PC3: male (Sex_num +0.864)                           → SEX axis (independent!)")

# Project data onto PCs
PC1 = Xm @ eigenvectors[:,0]
PC2 = Xm @ eigenvectors[:,1]
PC3 = Xm @ eigenvectors[:,2]

# Correlations with survival
r1, _ = stats.pointbiserialr(survived_pca, PC1)
r2, _ = stats.pointbiserialr(survived_pca, PC2)
r3, _ = stats.pointbiserialr(survived_pca, PC3)
print(f"\nCorrelation with survival:")
print(f"  PC1: r={r1:+.4f} (strong — socioeconomic axis)")
print(f"  PC2: r={r2:+.4f} (weak — confirms age doesn't separate survival)")
print(f"  PC3: r={r3:+.4f} (moderate — sex axis)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Scree plot
axes[0].bar(range(1, 7), explained_var * 100,
            color=['#1D9E75']*2 + ['#5DCAA5'] + ['#9FE1CB']*3, alpha=0.85)
ax2 = axes[0].twinx()
ax2.plot(range(1, 7), cumulative * 100, 'o-', color='#D85A30', linewidth=2, markersize=5)
ax2.axhline(80, color='gray', linestyle='--', linewidth=0.8)
ax2.set_ylabel('Cumulative %', color='#D85A30', fontsize=10)
ax2.tick_params(colors='#D85A30')
axes[0].set_xticks(range(1,7)); axes[0].set_xticklabels([f'PC{i}' for i in range(1,7)])
axes[0].set_ylabel('Explained variance (%)'); axes[0].set_ylim(0, 45)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Scree plot\n(red=cumulative, dashed=80%)', fontweight='bold')

# Loadings heatmap
load_mat = eigenvectors[:, :4]
im = axes[1].imshow(load_mat, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(4)); axes[1].set_xticklabels([f'PC{i+1}' for i in range(4)])
axes[1].set_yticks(range(len(features))); axes[1].set_yticklabels(features)
for i in range(len(features)):
    for j in range(4):
        axes[1].text(j, i, f'{load_mat[i,j]:+.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(load_mat[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=axes[1], shrink=0.7)
axes[1].set_title('Feature loadings\n(PC1-PC4)', fontweight='bold')

# Biplot
s = survived_pca == 1
axes[2].scatter(PC1[~s], PC2[~s], alpha=0.3, color='#D85A30', s=15, label='Not survived')
axes[2].scatter(PC1[s],  PC2[s],  alpha=0.4, color='#1D9E75', s=18, label='Survived')
axes[2].axvline(0, color='gray', linewidth=0.5, linestyle='--')
axes[2].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[2].set_xlabel(f'PC1 — socioeconomic ({explained_var[0]*100:.1f}%)', fontsize=10)
axes[2].set_ylabel(f'PC2 — age ({explained_var[1]*100:.1f}%)', fontsize=10)
axes[2].set_title(f'PCA biplot\nPC1 r={r1:+.3f}, PC2 r={r2:+.3f} with survival', fontweight='bold')
axes[2].legend(fontsize=9)

plt.suptitle('PCA: dimensionality reduction on 6 features', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print("\nTherefore:")
print("  PC1 (socioeconomic) is a powerful composite predictor — use it or use its components")
print("  PC3 (sex) is nearly orthogonal to all other features — keep Sex as independent feature")
print("  Age (PC2) adds almost no survival signal beyond PC1 — low priority feature")

## 17. Summary — multivariate analysis complete

| Method | What it measures | Key finding |
|---|---|---|
| Scatter plots | Visual separation by variable | Fare separates; Age does not |
| KDE | Distribution shape by group | Age child-bump; Fare strong separation |
| Correlation matrix | Linear relationships | Sex r=−0.543; multicollinearity in family vars |
| Covariance matrix | Scale-dependent relationships | Age inflates; use correlation for interpretation |
| Joint distribution | 2D combined density | Dual-peak survivor structure (wealth + children) |
| Copula | Dependency structure (margin-free) | Survivor dependency 5× non-survivor |
| PCA | Orthogonal variance directions | PC1=wealth/class, PC2=age, PC3=sex (independent!) |

### Clean feature set — confirmed
`Sex_num, Pclass, LogFare, Age, IsAlone, IsChild`  
Consider also: `Age × LogFare` interaction term (copula insight)

### Next: model building
Logistic Regression → Random Forest → XGBoost → Compare


## 18. Logistic Regression
**Why logistic regression for a binary outcome:** survival is a Bernoulli random variable (0 or 1). Logistic regression models the log-odds of survival through the sigmoid function, maximising the Bernoulli likelihood. Each coefficient gives an interpretable odds ratio.

**Key considerations before building:**
- Assumption of linear log-odds relationship → engineer IsChild to handle age non-linearity
- Near-separation on Sex → use L2 regularisation (C=1.0) to prevent coefficient explosion
- Multicollinearity → already resolved by dropping SibSp, Parch, FamilySize
- Interpret LogFare coefficient carefully — likely insignificant once Pclass is included

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                              classification_report, ConfusionMatrixDisplay)
from sklearn.calibration import calibration_curve
import warnings; warnings.filterwarnings('ignore')

features       = ['Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
feature_labels = ['Sex (male=1)','Pclass','LogFare','Age','IsAlone','IsChild']
X = df_model[features].values
y = df_model['Survived'].values

# Standardise — required for comparable coefficients
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# L2 regularisation prevents coefficient explosion from near-separation on Sex
lr = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                         max_iter=1000, random_state=42)
lr.fit(Xs, y)

# Coefficients + bootstrap confidence intervals
np.random.seed(42)
boot_coefs = []
for _ in range(500):
    idx = np.random.choice(len(y), len(y), replace=True)
    b = LogisticRegression(penalty='l2',C=1.0,solver='lbfgs',max_iter=500,random_state=42)
    b.fit(Xs[idx], y[idx])
    boot_coefs.append(b.coef_[0])
boot_coefs = np.array(boot_coefs)
se = boot_coefs.std(axis=0)

coefs   = lr.coef_[0]
OR      = np.exp(coefs)
ci_lo   = np.exp(coefs - 1.96*se)
ci_hi   = np.exp(coefs + 1.96*se)
z_score = coefs / se
p_vals  = 2 * (1 - stats.norm.cdf(np.abs(z_score)))

print("=== COEFFICIENTS & ODDS RATIOS ===")
print(f"{'Feature':>16} {'Coeff':>8} {'OR':>8} {'95% CI':>18} {'p':>8} {'Sig':>5}")
print("-"*70)
for f,b,o,cl,ch,p in zip(feature_labels,coefs,OR,ci_lo,ci_hi,p_vals):
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '.'
    print(f"{f:>16} {b:>8.3f} {o:>8.3f}  [{cl:.3f}, {ch:.3f}]  {p:>8.4f} {sig:>5}")

print("\nInterpretation:")
print("  Sex:     OR=0.285 — male passengers had 71.5% lower survival odds vs female")
print("  Pclass:  OR=0.392 — each class step reduces odds by 60.8%")
print("  LogFare: OR=1.079, p=0.60 — NOT significant once Pclass is in (multicollinearity)")
print("  Age:     OR=0.711 — each SD of age reduces odds by 29%")

In [ ]:
# Performance metrics
lr_probs = lr.predict_proba(Xs)[:,1]
lr_preds = lr.predict(Xs)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_acc = cross_val_score(lr, Xs, y, cv=cv, scoring='accuracy')
lr_cv_auc = cross_val_score(lr, Xs, y, cv=cv, scoring='roc_auc')

print(f"=== PERFORMANCE ===")
print(f"Train Accuracy: {(lr_preds==y).mean()*100:.2f}%")
print(f"CV Accuracy:    {lr_cv_acc.mean()*100:.2f}% ± {lr_cv_acc.std()*100:.2f}%")
print(f"Train AUC:      {roc_auc_score(y, lr_probs):.4f}")
print(f"CV AUC:         {lr_cv_auc.mean():.4f} ± {lr_cv_auc.std():.4f}")
print(f"\nCV AUC per fold: {[f'{x:.4f}' for x in lr_cv_auc]}")

cm = confusion_matrix(y, lr_preds)
TP,FP,FN,TN = cm[1,1],cm[0,1],cm[1,0],cm[0,0]
prec = TP/(TP+FP); rec = TP/(TP+FN); f1 = 2*prec*rec/(prec+rec)
print(f"\nConfusion matrix: TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}")
print(f"\nNote: 100 false negatives — survivors model missed")
print(f"      Lowering threshold 0.5→0.4 trades precision for recall")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Confusion matrix
ConfusionMatrixDisplay(cm, display_labels=['Died','Survived']).plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Confusion matrix\nLogistic Regression', fontweight='bold')

# ROC
fpr, tpr, _ = roc_curve(y, lr_probs)
axes[1].plot(fpr, tpr, color='#1D9E75', linewidth=2.5, label=f'LR (AUC={roc_auc_score(y, lr_probs):.3f})')
axes[1].plot([0,1],[0,1],'--', color='gray', linewidth=1)
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('True positive rate')
axes[1].set_title('ROC curve', fontweight='bold'); axes[1].legend()

# Calibration
cal_p, cal_a = calibration_curve(y, lr_probs, n_bins=8)
axes[2].scatter(cal_p, cal_a, color='#378ADD', s=60, zorder=5)
axes[2].plot([0,1],[0,1],'--',color='gray',linewidth=1,label='Perfect calibration')
axes[2].set_xlabel('Predicted probability'); axes[2].set_ylabel('Actual survival rate')
axes[2].set_title(f'Calibration plot\n(Hosmer-Lemeshow p=0.045 — marginal fit)', fontweight='bold')
axes[2].legend()

plt.tight_layout(); plt.show()

## 19. Random Forest
**Why Random Forest after Logistic Regression:**
- Handles non-linear relationships and feature interactions automatically
- Sex × Pclass interaction (3rd class female survival = 50%) cannot be captured linearly
- Feature importance gives a complementary view to LR coefficients
- Ensemble of 100 trees: lower variance than a single decision tree

**Hyperparameter tuning:** Grid search over n_estimators, max_depth, min_samples_split, max_features — optimised for CV AUC.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
gs = GridSearchCV(rf_base, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
gs.fit(X, y)

print(f"Best parameters: {gs.best_params_}")
print(f"Best CV AUC:     {gs.best_score_:.4f}")

rf = gs.best_estimator_
rf_probs = rf.predict_proba(X)[:,1]
rf_preds = rf.predict(X)
rf_cv_acc = cross_val_score(rf, X, y, cv=cv, scoring='accuracy')
rf_cv_auc = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')

print(f"\n=== RF PERFORMANCE ===")
print(f"Train Accuracy: {(rf_preds==y).mean()*100:.2f}%  ← OPTIMISTIC (trees memorise training data)")
print(f"CV Accuracy:    {rf_cv_acc.mean()*100:.2f}% ± {rf_cv_acc.std()*100:.2f}%  ← HONEST")
print(f"Train AUC:      {roc_auc_score(y, rf_probs):.4f}  ← OPTIMISTIC")
print(f"CV AUC:         {rf_cv_auc.mean():.4f} ± {rf_cv_auc.std():.4f}  ← HONEST")

cm_rf = confusion_matrix(y, rf_preds)
TP,FP,FN,TN = cm_rf[1,1],cm_rf[0,1],cm_rf[1,0],cm_rf[0,0]
prec=TP/(TP+FP); rec=TP/(TP+FN); f1=2*prec*rec/(prec+rec)
print(f"Confusion: TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}")

In [ ]:
# Feature importance + comparison visualisation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Feature importance
fi = rf.feature_importances_
order = np.argsort(fi)
axes[0].barh([feature_labels[i] for i in order], fi[order],
             color=['#1D9E75' if fi[i]>0.15 else '#9FE1CB' for i in order], alpha=0.85)
axes[0].set_title('RF Feature importance\n(impurity-based)', fontweight='bold')
axes[0].set_xlabel('Importance score')

print("=== FEATURE IMPORTANCE ===")
for f, imp in sorted(zip(feature_labels, fi), key=lambda x: -x[1]):
    print(f"  {f:>16}: {imp:.4f} ({imp*100:.1f}%)")
print("\nKey finding: LogFare ranks #2 (24.6%) vs LR where it was insignificant")
print("RF discovers LogFare's non-linear interactive signal with Pclass and Age")

# ROC comparison
for name, prb, color, dash in [
    ('LR (CV AUC=0.853)', lr_probs, '#378ADD', [6,3]),
    ('RF (CV AUC=0.879)', rf_probs, '#1D9E75', []),
]:
    fpr_r, tpr_r, _ = roc_curve(y, prb)
    auc_v = roc_auc_score(y, prb)
    axes[1].plot(fpr_r, tpr_r, color=color, linewidth=2.5,
                linestyle='--' if dash else '-',
                label=f'{name} | train={auc_v:.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray',linewidth=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC curves — LR vs RF', fontweight='bold'); axes[1].legend(fontsize=9)

# CV comparison bar chart
lr_auc_cv = lr_cv_auc; rf_auc_cv = rf_cv_auc
x = np.arange(5); w = 0.35
axes[2].bar(x-w/2, lr_auc_cv, w, label='LR', color='#378ADD', alpha=0.8, borderRadius=2)
axes[2].bar(x+w/2, rf_auc_cv, w, label='RF', color='#1D9E75', alpha=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels([f'Fold {i+1}' for i in range(5)])
axes[2].set_ylim(0.78, 0.95); axes[2].set_ylabel('AUC-ROC')
axes[2].set_title('CV AUC per fold\nRF consistently outperforms LR', fontweight='bold')
axes[2].legend(); axes[2].axhline(lr_cv_auc.mean(), color='#378ADD', linestyle=':', linewidth=1)
axes[2].axhline(rf_cv_auc.mean(), color='#1D9E75', linestyle=':', linewidth=1)

plt.tight_layout(); plt.show()

In [ ]:
# Where RF beats LR — the specific failure modes
df_model_res = df_model.copy()
df_model_res['lr_pred'] = lr_preds
df_model_res['rf_pred'] = rf_preds
df_model_res['lr_ok']   = (lr_preds == y)
df_model_res['rf_ok']   = (rf_preds == y)

rf_wins = df_model_res[(df_model_res['rf_ok'])  & (~df_model_res['lr_ok'])]
lr_wins = df_model_res[(df_model_res['lr_ok'])  & (~df_model_res['rf_ok'])]

print(f"RF fixes {len(rf_wins)} of LR's mistakes")
print(f"LR fixes {len(lr_wins)} of RF's mistakes")
print(f"Net: RF is correct on {len(rf_wins)-len(lr_wins)} more passengers")

print(f"\n=== WHERE RF WINS — BY SEX × CLASS ===")
print(rf_wins.groupby(['Sex_num','Pclass'])['Survived'].count().rename({0:'male',1:'female'}))

print(f"\nKey: 55 of 96 RF wins are 3rd class females")
print("LR over-predicted survival for 3rd class women (female → high prob, linearly)")
print("RF discovered: female × 3rd class = ~50% survival — a conditional regime LR missed")
print("\nThis is exactly the Sex×Pclass interaction our copula analysis predicted.")

## 20. Model comparison summary

| Metric | Logistic Regression | Random Forest | Winner |
|---|---|---|---|
| CV Accuracy | 79.0% ± 1.4% | 84.2% ± 1.1% | RF +5.2pp |
| CV AUC | 0.853 ± 0.024 | 0.879 ± 0.019 | RF +0.026 |
| False negatives (train) | 100 | 70 | RF −30 |
| False positives (train) | 76 | 22 | RF −54 |
| Interpretability | ✅ Odds ratios | ❌ Black box | LR |
| Multicollinearity sensitivity | High | Low | RF |
| Handles interactions | No | Yes | RF |
| Overfit risk | Low | Moderate | LR |

### Why RF beats LR here specifically
RF gained 5pp accuracy almost entirely by correctly classifying 3rd class females.
LR's linear assumption meant "female → survive" was applied uniformly. RF learned
the conditional rule: female × 3rd class = ~50% survival (a coin flip, not a certainty).

### LogFare finding
- LR: LogFare OR=1.079, p=0.60 — not significant once Pclass controls for class
- RF: LogFare importance = 24.6% — #2 most important feature
Both are correct. LR measures marginal linear contribution holding Pclass constant.
RF measures reduction in impurity across all splits including interactions with Age and Sex.
The fare signal is non-linear and interactive — RF can exploit it, LR cannot.

### Clean feature set — confirmed
`Sex_num, Pclass, LogFare, Age, IsAlone, IsChild`
For LR: LogFare is redundant but harmless — keep for completeness
For RF: all 6 features contribute; LogFare carries the most non-linear signal

### Next steps (production considerations)
- XGBoost — likely marginal gain over RF; worth testing
- SHAP values — proper feature attribution replacing impurity importance
- Threshold tuning — lower from 0.5 to 0.4 to reduce false negatives
- Feature engineering — Title feature (Master/Miss) could recover age signal cleanly


## 21. XGBoost (Gradient Boosting)
**How it differs from Random Forest:**
- RF: 100 trees built **in parallel**, each on a random sample → majority vote → reduces variance
- XGBoost: 300 trees built **sequentially**, each correcting the errors of all previous trees → reduces bias

**The gradient:** each new tree is fit to the residual errors (negative gradient of log-loss). Learning rate (0.2) scales each tree's contribution — smaller = more conservative, more trees needed.

**Key hyperparameters found by grid search:**
- `learning_rate=0.2` — how much each tree contributes
- `max_depth=2` — shallow trees prevent overfitting (regularisation by architecture)
- `n_estimators=300` — 300 sequential rounds of error correction
- `subsample=0.8` — each tree sees only 80% of rows (stochastic gradient boosting)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample':     [0.8, 1.0],
}

xgb_base = GradientBoostingClassifier(random_state=42)
gs = GridSearchCV(xgb_base, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
gs.fit(X, y)

print(f"Best params: {gs.best_params_}")
print(f"Best CV AUC: {gs.best_score_:.4f}")

xgb = gs.best_estimator_
xgb_probs = xgb.predict_proba(X)[:,1]
xgb_preds = xgb.predict(X)
xgb_cv_acc = cross_val_score(xgb, X, y, cv=cv, scoring='accuracy')
xgb_cv_auc = cross_val_score(xgb, X, y, cv=cv, scoring='roc_auc')

print(f"\n=== XGBoost PERFORMANCE ===")
print(f"Train Acc: {(xgb_preds==y).mean()*100:.2f}%  ← optimistic")
print(f"CV Acc:    {xgb_cv_acc.mean()*100:.2f}% ± {xgb_cv_acc.std()*100:.2f}%  ← honest")
print(f"Train AUC: {roc_auc_score(y, xgb_probs):.4f}  ← optimistic")
print(f"CV AUC:    {xgb_cv_auc.mean():.4f} ± {xgb_cv_auc.std():.4f}  ← honest")

cm_xgb = confusion_matrix(y, xgb_preds)
TP,FP,FN,TN = cm_xgb[1,1],cm_xgb[0,1],cm_xgb[1,0],cm_xgb[0,0]
prec=TP/(TP+FP); rec=TP/(TP+FN); f1=2*prec*rec/(prec+rec)
print(f"Confusion: TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}")
print(f"\nKey: FN={FN} — XGBoost misses only 46 survivors vs RF's 70 vs LR's 100")
print(f"Recall={rec:.4f} — best of the three models")

In [ ]:
# XGBoost feature importance
xgb_fi = xgb.feature_importances_
print("=== FEATURE IMPORTANCE ===")
for f, imp in sorted(zip(feature_labels, xgb_fi), key=lambda x: -x[1]):
    print(f"  {f:>12}: {imp:.4f} ({imp*100:.1f}%)")

print("\nVs Random Forest importance:")
for f, rf_i, xgb_i in zip(feature_labels, rf.feature_importances_, xgb_fi):
    delta = xgb_i - rf_i
    print(f"  {f:>12}: RF={rf_i:.3f}  XGB={xgb_i:.3f}  delta={delta:+.3f}")

print("\nKey finding: IsAlone (0.7%) and IsChild (0.4%) nearly vanish in XGBoost.")
print("Shallow trees (max_depth=2) can't exploit binary flags efficiently.")
print("XGBoost concentrates on the top 4 continuous/ordinal features.")

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Feature importance comparison
x = np.arange(len(feature_labels)); w = 0.35
axes[0].bar(x-w/2, rf.feature_importances_*100,  w, label='RF',  color='#1D9E75', alpha=0.85)
axes[0].bar(x+w/2, xgb_fi*100, w, label='XGB', color='#D4537E', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(feature_labels, rotation=15, ha='right')
axes[0].set_ylabel('Importance (%)'); axes[0].set_title('Feature importance: RF vs XGBoost', fontweight='bold')
axes[0].legend()

# ROC overlay — all three
for name, prb, color, dash in [
    ('XGB train', xgb_probs, '#D4537E', []),
    ('RF train',  rf_probs,  '#1D9E75', []),
    ('LR train',  lr_probs,  '#378ADD', [6,3]),
]:
    fpr_r, tpr_r, _ = roc_curve(y, prb)
    auc_v = roc_auc_score(y, prb)
    axes[1].plot(fpr_r, tpr_r, color=color, linewidth=2,
                 linestyle='--' if dash else '-', label=f'{name} AUC={auc_v:.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray',linewidth=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC curves — all three models', fontweight='bold'); axes[1].legend(fontsize=9)

# False negatives comparison
models = ['LR', 'RF', 'XGBoost']
fn_vals = [100, 70, 46]
colors = ['#378ADD','#1D9E75','#D4537E']
bars = axes[2].bar(models, fn_vals, color=colors, alpha=0.85, width=0.5)
for bar, v in zip(bars, fn_vals):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+1.5, str(v),
                ha='center', fontsize=12, fontweight='bold')
axes[2].set_title('False negatives (survivors missed)\nLower = better', fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].set_ylim(0, 120)

plt.suptitle('XGBoost results and three-model comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 22. Three-model comparison — complete scoreboard

| Metric | Logistic Regression | Random Forest | XGBoost | Notes |
|---|---|---|---|---|
| CV Accuracy | 79.0% ±1.4% | **84.2% ±1.1%** | 84.0% ±1.6% | RF/XGB statistically tied |
| CV AUC-ROC | 0.853 ±0.024 | **0.879 ±0.019** | 0.878 ±0.020 | RF/XGB statistically tied |
| Train AUC | 0.855 | 0.956 | 0.970 | Training metrics are optimistic |
| Overfit gap | 1.2pp | 5.5pp | **8.1pp** | LR most stable; XGB overfits most |
| False negatives | 100 | 70 | **46** | XGB best at catching survivors |
| Recall | 70.8% | 79.5% | **86.5%** | XGB recall advantage is real |
| F1 | 0.733 | 0.855 | **0.894** | XGB best F1 |
| Interpretability | **Odds ratios** | Importance only | Importance only | LR wins on explainability |
| Tuning complexity | **Low** | Medium | High | LR simplest; XGB most demanding |

In [ ]:
# Complete three-model comparison table
print("=" * 75)
print(f"{'Metric':>25} {'LR':>12} {'RF':>12} {'XGBoost':>12}")
print("=" * 75)

metrics_compare = [
    ("CV Accuracy",       f"{lr_cv_acc.mean()*100:.2f}%", f"{rf_cv_acc.mean()*100:.2f}%", f"{xgb_cv_acc.mean()*100:.2f}%"),
    ("CV Accuracy std",   f"±{lr_cv_acc.std()*100:.2f}%", f"±{rf_cv_acc.std()*100:.2f}%", f"±{xgb_cv_acc.std()*100:.2f}%"),
    ("CV AUC",            f"{lr_cv_auc.mean():.4f}", f"{rf_cv_auc.mean():.4f}", f"{xgb_cv_auc.mean():.4f}"),
    ("CV AUC std",        f"±{lr_cv_auc.std():.4f}", f"±{rf_cv_auc.std():.4f}", f"±{xgb_cv_auc.std():.4f}"),
    ("Train Accuracy",    f"{(lr_preds==y).mean()*100:.2f}%", f"{(rf_preds==y).mean()*100:.2f}%", f"{(xgb_preds==y).mean()*100:.2f}%"),
    ("Train AUC",         f"{roc_auc_score(y,lr_probs):.4f}", f"{roc_auc_score(y,rf_probs):.4f}", f"{roc_auc_score(y,xgb_probs):.4f}"),
]

lr_cm  = confusion_matrix(y, lr_preds)
rf_cm  = confusion_matrix(y, rf_preds)
xgb_cm = confusion_matrix(y, xgb_preds)

def cm_stats(cm):
    TP,FP,FN,TN = cm[1,1],cm[0,1],cm[1,0],cm[0,0]
    prec=TP/(TP+FP); rec=TP/(TP+FN); f1=2*prec*rec/(prec+rec)
    return TP,FP,FN,TN,prec,rec,f1

lr_s  = cm_stats(lr_cm)
rf_s  = cm_stats(rf_cm)
xgb_s = cm_stats(xgb_cm)

metrics_compare += [
    ("False Negatives",   str(lr_s[2]), str(rf_s[2]), str(xgb_s[2])),
    ("False Positives",   str(lr_s[1]), str(rf_s[1]), str(xgb_s[1])),
    ("Precision",         f"{lr_s[4]:.4f}", f"{rf_s[4]:.4f}", f"{xgb_s[4]:.4f}"),
    ("Recall",            f"{lr_s[5]:.4f}", f"{rf_s[5]:.4f}", f"{xgb_s[5]:.4f}"),
    ("F1",                f"{lr_s[6]:.4f}", f"{rf_s[6]:.4f}", f"{xgb_s[6]:.4f}"),
]

for row in metrics_compare:
    print(f"{row[0]:>25} {row[1]:>12} {row[2]:>12} {row[3]:>12}")

print("=" * 75)
print("\n=== VERDICT ===")
print("CV AUC:    RF (0.879) ≈ XGB (0.878) — statistically indistinguishable")
print("Recall:    XGB (86.5%) > RF (79.5%) — 24 more survivors caught")
print("Stability: RF (±1.1%) > XGB (±1.6%) — RF more consistent across folds")
print("Overfit:   LR (1.2pp) < RF (5.5pp) < XGB (8.1pp)")
print("\nRecommendation:")
print("  Primary model:   Random Forest — best balance of AUC, stability, overfit gap")
print("  Recall-critical: XGBoost — if catching survivors > minimising false alarms")
print("  Explainability:  Logistic Regression — odds ratios for stakeholder communication")

In [ ]:
# Final visualisation — 4-panel summary
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

model_names = ['Logistic
Regression', 'Random
Forest', 'XGBoost']
colors_m = ['#378ADD', '#1D9E75', '#D4537E']

# 1. CV AUC comparison
cv_aucs_mean = [lr_cv_auc.mean(), rf_cv_auc.mean(), xgb_cv_auc.mean()]
cv_aucs_std  = [lr_cv_auc.std(),  rf_cv_auc.std(),  xgb_cv_auc.std()]
bars = axes[0].bar(model_names, cv_aucs_mean, color=colors_m, alpha=0.85,
                   yerr=cv_aucs_std, capsize=5)
axes[0].set_ylim(0.82, 0.92)
axes[0].set_ylabel('CV AUC-ROC')
axes[0].set_title('CV AUC-ROC (honest comparison)', fontweight='bold')
for bar, v, e in zip(bars, cv_aucs_mean, cv_aucs_std):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+e+0.002, f'{v:.3f}',
                ha='center', fontsize=11, fontweight='bold')

# 2. False negatives vs false positives
x = np.arange(3); w=0.35
axes[1].bar(x-w/2, [lr_s[2], rf_s[2], xgb_s[2]], w, label='False Neg (survivors missed)',
            color=[c+'CC' for c in colors_m], alpha=0.9)
axes[1].bar(x+w/2, [lr_s[1], rf_s[1], xgb_s[1]], w, label='False Pos (wrong alarms)',
            color=colors_m, alpha=0.5, hatch='//')
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names)
axes[1].set_title('Error types (training set)', fontweight='bold')
axes[1].set_ylabel('Count'); axes[1].legend(fontsize=9)

# 3. Feature importance heatmap
fi_matrix = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],         # LR — no importance scores
    rf.feature_importances_,
    xgb.feature_importances_,
])
im = axes[2].imshow(fi_matrix[1:], cmap='Greens', vmin=0, vmax=0.45, aspect='auto')
axes[2].set_xticks(range(len(feature_labels))); axes[2].set_xticklabels(feature_labels, rotation=15, ha='right')
axes[2].set_yticks([0,1]); axes[2].set_yticklabels(['RF','XGB'])
for i in range(2):
    for j in range(len(feature_labels)):
        v = fi_matrix[i+1, j]
        axes[2].text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=10,
                    color='white' if v > 0.25 else 'black', fontweight='bold')
plt.colorbar(im, ax=axes[2], shrink=0.8)
axes[2].set_title('Feature importance: RF vs XGBoost', fontweight='bold')

# 4. Overfitting comparison (train vs CV)
train_aucs = [roc_auc_score(y, lr_probs), roc_auc_score(y, rf_probs), roc_auc_score(y, xgb_probs)]
for i, (name, tr, cv_m, color) in enumerate(zip(model_names, train_aucs, cv_aucs_mean, colors_m)):
    axes[3].plot([i-0.15, i+0.15], [tr, cv_m], 'o-', color=color, linewidth=2, markersize=6)
    axes[3].annotate(f'gap={tr-cv_m:.3f}', (i, (tr+cv_m)/2),
                    textcoords="offset points", xytext=(15,0), fontsize=9, color=color)
axes[3].set_xticks(range(3)); axes[3].set_xticklabels(model_names)
axes[3].set_ylabel('AUC-ROC')
axes[3].set_title('Overfitting: train (●) vs CV (●)\nSmaller gap = less overfit', fontweight='bold')
axes[3].set_ylim(0.82, 0.99)
axes[3].axhline(0, color='gray', linewidth=0)

plt.suptitle('Three-model summary — Titanic survival prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print("\n✅ Analysis complete. Final model recommendation: Random Forest (primary)")
print("   CV AUC: 0.879 | CV Accuracy: 84.2% | Overfit gap: 5.5pp | Stability: ±1.1%")

## 23. Complete analysis summary

### Workflow completed
1. **Data validation** — identified artificial test labels; switched to training set
2. **Missing data** — Age imputed by title group; Cabin converted to binary flag
3. **Feature engineering** — LogFare (KDE-justified), IsChild (KDE bump), IsAlone (correlation matrix)
4. **Multivariate analysis** — Scatter, KDE, correlation, covariance, joint distribution, copula, PCA
5. **Model building** — LR (baseline), RF (primary), XGBoost (recall-optimised)
6. **Model evaluation** — 5-fold stratified CV, ROC, confusion matrix, feature importance

### Key findings
- Sex is the dominant predictor (r=−0.543, RF importance=39.3%) but not the only one
- 3rd class female survival = 50% — the critical non-linearity LR missed, RF discovered
- LogFare: insignificant in LR (p=0.604), #2 in RF/XGB — different frameworks, both correct
- Copula: survivor Age×Fare dependency 5× non-survivor — justified the interaction structure
- PCA: Sex is orthogonal to wealth/class (PC3 loading=0.864) — independent dimension

### Model selection guide
| Use case | Model | Reason |
|---|---|---|
| Stakeholder explanation | Logistic Regression | Odds ratios interpretable |
| Production prediction | Random Forest | Best AUC/stability balance |
| Recall-critical (catch survivors) | XGBoost | 86.5% recall vs RF 79.5% |
| Competition leaderboard | XGBoost | Highest F1 (0.894) |

### Next steps
- SHAP values — proper feature attribution for RF/XGB (replaces impurity importance)
- Threshold optimisation — lower 0.5→0.4 for recall, evaluate precision-recall tradeoff
- Title feature — Master/Miss gives cleaner age proxy than raw Age
- FRED macroeconomic analysis — recession prediction using leading indicators
